# Notebook 4: Demo streaming dự đoán real-time

Mô phỏng luồng dữ liệu RSSI và hiển thị kết quả trực quan.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
from pathlib import Path
import json
import time
import pandas as pd
import requests

from smarthome.config import load_config
from smarthome.utils import set_seed

In [ ]:
ROOT = Path(__file__).resolve().parents[1]
cfg = load_config(ROOT / "config.yaml")
BASE_URL = "http://127.0.0.1:5000"

In [ ]:
# Tải dữ liệu mẫu từ synthetic
sample = pd.read_csv(ROOT / "data" / "processed" / "room_test.csv")
metadata_cols = {"timestamp", "session_id", "device_id", "room", "is_anomaly", "anomaly_type", "confidence"}
ap_cols = [c for c in sample.columns if c not in metadata_cols]
print("Số AP mẫu:", len(ap_cols))

In [ ]:
import random
random.seed(42)

for _ in range(10):
    row = sample.sample(1).iloc[0]
    wifi = {ap: float(row.get(ap, -100.0)) for ap in ap_cols}
    payload = {
        "wifi": wifi,
        "timestamp": float(row.get("timestamp", time.time())),
    }
    r = requests.post(f"{BASE_URL}/api/scan", json=payload)
    data = r.json()
    print(data["room"], round(data["confidence"], 3), data["status"])